# 01b — Saraga Hindustani: Exploratory Data Analysis

This notebook characterises the processed Hindustani subset (60 Basic-Pitch
transcribed MIDI files). It covers the same five dimensions as 01a plus
tradition-specific analysis: raga and taal label distributions.

**Prerequisite:** Run `00b_hindustani_prep.ipynb` first (overnight transcription job).
A check cell below will raise a clear error if the MIDI files are not yet present.


In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.midi_utils import (
    load_midi, analyse_midi, get_pitch_class_histogram,
    pitch_class_entropy, piano_roll_plot
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

PC_LABELS = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [2]:
def build_stats_df(midi_dir, meta_df=None, id_col=None):
    """Analyse all MIDI files in midi_dir; optionally merge metadata."""
    midi_files = sorted(midi_dir.glob("*.mid")) + sorted(midi_dir.glob("*.midi"))
    print(f"Analysing {len(midi_files)} MIDI files ...")
    records = [analyse_midi(p) for p in midi_files]
    df = pd.DataFrame(records)
    if "error" in df.columns:
        bad = df["error"].notna().sum()
        if bad:
            print(f"  Warning: {bad} files failed to load.")
        df = df[df["error"].isna()].drop(columns=["error"])
    df["filename"] = [Path(p).name for p in df["path"]]
    if meta_df is not None and id_col is not None:
        df = df.merge(meta_df, left_on="filename", right_on=id_col, how="left")
    return df

In [3]:
def summary_panel(df, tradition_name, save_path):
    """4-panel summary figure: duration, note density, pitch range, PC entropy."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"{tradition_name} — EDA Summary (n={len(df)})", fontsize=13, y=1.01)

    # Duration
    ax = axes[0, 0]
    df["duration_s"].div(60).plot.hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
    ax.axvline(df["duration_s"].mean()/60, color="red", linestyle="--", label=f"mean={df['duration_s'].mean()/60:.1f} min")
    ax.set_xlabel("Duration (minutes)")
    ax.set_title("Duration Distribution")
    ax.legend(fontsize=8)

    # Note density
    ax = axes[0, 1]
    df["note_density"].plot.hist(bins=25, ax=ax, color="seagreen", edgecolor="white")
    ax.axvline(df["note_density"].mean(), color="red", linestyle="--", label=f"mean={df['note_density'].mean():.2f}")
    ax.set_xlabel("Notes per second")
    ax.set_title("Note Density Distribution")
    ax.legend(fontsize=8)

    # Pitch range
    ax = axes[1, 0]
    df["pitch_range"].plot.hist(bins=25, ax=ax, color="darkorange", edgecolor="white")
    ax.axvline(df["pitch_range"].mean(), color="red", linestyle="--", label=f"mean={df['pitch_range'].mean():.1f}")
    ax.set_xlabel("Pitch range (semitones)")
    ax.set_title("Pitch Range Distribution")
    ax.legend(fontsize=8)

    # PC entropy
    ax = axes[1, 1]
    df["pc_entropy"].plot.hist(bins=25, ax=ax, color="mediumpurple", edgecolor="white")
    ax.axvline(df["pc_entropy"].mean(), color="red", linestyle="--", label=f"mean={df['pc_entropy'].mean():.3f}")
    ax.set_xlabel("Pitch Class Entropy (bits)")
    ax.set_title("PC Entropy Distribution\n(Yang & Lerch, 2020)")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")


def mean_pc_histogram_plot(df_midi_paths, tradition_name, save_path):
    """Plot the mean pitch class histogram across all pieces."""
    hists = []
    for p in df_midi_paths:
        pm = load_midi(p)
        if pm:
            hists.append(get_pitch_class_histogram(pm))
    if not hists:
        return
    mean_hist = np.mean(hists, axis=0)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(PC_LABELS, mean_hist, color="steelblue", edgecolor="white")
    ax.set_xlabel("Pitch class")
    ax.set_ylabel("Mean relative frequency")
    ax.set_title(f"{tradition_name} — Mean Pitch Class Histogram (n={len(hists)})")
    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")

In [4]:
MIDI_DIR = PROJECT_ROOT / "data" / "processed" / "hindustani" / "midi"
META_CSV = PROJECT_ROOT / "data" / "metadata" / "hindustani_tracks.csv"

midi_files = list(MIDI_DIR.glob("*.mid"))
if not midi_files:
    raise FileNotFoundError(
        "No MIDI files found in data/processed/hindustani/midi/. "
        "Run notebook 00b_hindustani_prep.ipynb first (overnight job)."
    )
print(f"Found {len(midi_files)} MIDI files.")

meta = pd.read_csv(META_CSV)
print(f"Metadata rows: {len(meta)}")
meta[["title", "raga", "taal", "metadata_source"]].head(5)

Found 60 MIDI files.
Metadata rows: 60


,title,raga,taal,metadata_source
0,Raag Gavti,Gawti,Rupak,json
1,Raag Abhogi,Abhogi,Ektaal,json
2,Thumri in Piloo,Mishra piloo,Jatt,json
3,Bairagi,Bairagi,Ektaal,json
4,Nat Bhairon,Nat bhairav,Ektaal,json


## 1. Metadata quality

In [5]:
print("Metadata source breakdown:")
print(meta["metadata_source"].value_counts().to_string())
print(f"\nUnique ragas : {meta['raga'].nunique()}")
print(f"Unique taals : {meta['taal'].nunique()}")
print("\nTop 15 ragas in selection:")
print(meta["raga"].value_counts().head(15).to_string())

Metadata source breakdown:
metadata_source
json    60

Unique ragas : 53
Unique taals : 7

Top 15 ragas in selection:
raga
Bhairabi                 3
Lalat                    2
Shree                    2
Marwa                    2
Todi                     2
Jog                      2
Gawti                    1
Yaman kalyan             1
Jait Kalyan              1
Bibhas                   1
Miya malhar              1
Bahar                    1
Dagori                   1
Puriya dhanashree        1
Komal rishabh asavari    1


## 2. Compute statistics

In [6]:
stats = build_stats_df(MIDI_DIR)
print(f"Files analysed: {len(stats)}")
print("\nDescriptive statistics:")
print(stats[["duration_s", "note_count", "note_density", "pitch_range", "pc_entropy"]].describe().round(3))

Analysing 60 MIDI files ...
Files analysed: 60

Descriptive statistics:
       duration_s  note_count  note_density  pitch_range  pc_entropy
count      60.000      60.000        60.000       60.000      60.000
mean     1282.838    5937.217         4.735       58.817       2.745
std       957.429    4728.055         1.365        7.652       0.214
min        85.850     462.000         1.373       41.000       2.234
25%       588.483    2565.000         4.207       53.000       2.621
50%       894.660    4092.000         5.072       57.000       2.764
75%      1783.732    8632.500         5.314       65.000       2.912
max      4237.050   22712.000         8.688       74.000       3.179


In [7]:
# Note: Hindustani recordings are long concerts; durations will be large.
print(f"Duration range: {stats['duration_s'].min()/60:.1f} – {stats['duration_s'].max()/60:.1f} minutes")
print(f"Mean duration : {stats['duration_s'].mean()/60:.1f} minutes")

Duration range: 1.4 – 70.6 minutes
Mean duration : 21.4 minutes


## 3. Distribution plots

Note: High note density relative to Western Classical is expected — Basic-Pitch
transcribes all pitched sources in the mix (vocal + harmonium), producing denser
note sequences than single-instrument piano MIDI.


In [8]:
summary_panel(stats, "Hindustani Classical (Saraga)", RESULTS_DIR / "eda_hindustani_summary.png")

Saved → results/eda_hindustani_summary.png


## 4. Mean pitch class histogram

In [9]:
mean_pc_histogram_plot(stats["path"].tolist(), "Hindustani Classical (Saraga)",
                       RESULTS_DIR / "eda_hindustani_pc_histogram.png")

Saved → results/eda_hindustani_pc_histogram.png


## 5. Raga label distribution

The raga distribution reflects the coverage in the Saraga Hindustani 1.5 dataset;
we have not artificially balanced raga counts — the distribution is as-recorded.


In [10]:
meta_valid = meta[meta["midi_path"].notna()].copy() if "midi_path" in meta.columns else meta.copy()
raga_counts = meta_valid["raga"].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
raga_counts.head(20).plot(kind="bar", ax=ax, color="indianred", edgecolor="white")
ax.set_xlabel("Raga")
ax.set_ylabel("Number of tracks")
ax.set_title(f"Hindustani — Raga Distribution (top 20, n={len(meta_valid)})")
ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_hindustani_raga_dist.png"), dpi=150)
plt.show()

## 6. Sample piano roll

In [11]:
sample_path = stats["path"].iloc[0]
pm = load_midi(sample_path)
sample_name = Path(sample_path).stem[:60]

fig, ax = plt.subplots(figsize=(14, 4))
piano_roll_plot(pm, ax, time_start=60, time_end=90,
                title=f"Piano roll sample (60–90 s) — {sample_name}")
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_hindustani_piano_roll.png"), dpi=150)
plt.show()

## 7. Save summary statistics

In [12]:
stats["tradition"] = "hindustani"
stats.to_csv(RESULTS_DIR / "eda_hindustani_stats.csv", index=False)
print("Saved → results/eda_hindustani_stats.csv")
print(f"\nPC entropy mean : {stats['pc_entropy'].mean():.3f} bits")
print(f"Note density mean: {stats['note_density'].mean():.2f} notes/s")

Saved → results/eda_hindustani_stats.csv

PC entropy mean : 2.745 bits
Note density mean: 4.74 notes/s
